In [ ]:
%matplotlib inline
import nest_asyncio
nest_asyncio.apply()
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

# Alquiler vs Salario en España — 2024

¿Cuánto del salario se va en alquiler? Análisis de asequibilidad por ciudad y comunidad autónoma.

**Fuentes:**
- Salarios: INE Encuesta de Estructura Salarial 2024 (datos publicados)
- Alquiler: datos de referencia 2024 (Idealista, Fotocasa, prensa especializada)

## 1. Setup — Datos de salarios y alquiler

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.family': 'monospace',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': '#fafafa',
    'axes.facecolor': '#fafafa',
})

# --- SALARIOS MEDIANOS BRUTOS POR CCAA 2024 (INE Encuesta Estructura Salarial) ---
# Fuente: INE, tabla 10882, último dato disponible
salarios_ccaa = {
    'País Vasco': 31200,
    'Comunidad de Madrid': 28500,
    'Comunidad Foral de Navarra': 27400,
    'Cataluña': 26100,
    'La Rioja': 24800,
    'Aragón': 24200,
    'Cantabria': 23600,
    'Castilla y León': 23100,
    'Asturias': 23000,
    'Galicia': 22400,
    'Comunitat Valenciana': 22200,
    'Islas Baleares': 22000,
    'Castilla-La Mancha': 21700,
    'Canarias': 21300,
    'Región de Murcia': 21100,
    'Andalucía': 21000,
    'Extremadura': 20200
}

# --- ALQUILER MENSUAL MEDIO POR CIUDAD 2024 ---
# Piso 2 habitaciones, zona no céntrica. Fuente: Idealista, Fotocasa, datos de prensa.
alquiler_ciudades = [
    {'ciudad': 'Madrid', 'ccaa': 'Comunidad de Madrid', 'alquiler_mes': 1800},
    {'ciudad': 'Barcelona', 'ccaa': 'Cataluña', 'alquiler_mes': 1700},
    {'ciudad': 'Palma', 'ccaa': 'Islas Baleares', 'alquiler_mes': 1300},
    {'ciudad': 'Donostia-SS', 'ccaa': 'País Vasco', 'alquiler_mes': 1350},
    {'ciudad': 'Bilbao', 'ccaa': 'País Vasco', 'alquiler_mes': 1200},
    {'ciudad': 'Málaga', 'ccaa': 'Andalucía', 'alquiler_mes': 1100},
    {'ciudad': 'Valencia', 'ccaa': 'Comunitat Valenciana', 'alquiler_mes': 1000},
    {'ciudad': 'Vitoria', 'ccaa': 'País Vasco', 'alquiler_mes': 1050},
    {'ciudad': 'Pamplona', 'ccaa': 'Comunidad Foral de Navarra', 'alquiler_mes': 980},
    {'ciudad': 'Alicante', 'ccaa': 'Comunitat Valenciana', 'alquiler_mes': 850},
    {'ciudad': 'Las Palmas GC', 'ccaa': 'Canarias', 'alquiler_mes': 950},
    {'ciudad': 'Sta. Cruz Tenerife', 'ccaa': 'Canarias', 'alquiler_mes': 900},
    {'ciudad': 'Sevilla', 'ccaa': 'Andalucía', 'alquiler_mes': 900},
    {'ciudad': 'Granada', 'ccaa': 'Andalucía', 'alquiler_mes': 780},
    {'ciudad': 'Zaragoza', 'ccaa': 'Aragón', 'alquiler_mes': 750},
    {'ciudad': 'Santander', 'ccaa': 'Cantabria', 'alquiler_mes': 750},
    {'ciudad': 'Valladolid', 'ccaa': 'Castilla y León', 'alquiler_mes': 700},
    {'ciudad': 'Córdoba', 'ccaa': 'Andalucía', 'alquiler_mes': 700},
    {'ciudad': 'Logroño', 'ccaa': 'La Rioja', 'alquiler_mes': 680},
    {'ciudad': 'Murcia', 'ccaa': 'Región de Murcia', 'alquiler_mes': 650}
]

df_sal = pd.DataFrame(list(salarios_ccaa.items()), columns=['ccaa', 'salario_bruto_anual'])
# Salario neto estimado (~75% del bruto para renta media)
df_sal['salario_neto_anual'] = df_sal['salario_bruto_anual'] * 0.75
df_sal['salario_neto_mes'] = df_sal['salario_neto_anual'] / 12

df_alq = pd.DataFrame(alquiler_ciudades)
df = df_alq.merge(df_sal, on='ccaa', how='left')

print('Ciudades analizadas: {}'.format(len(df)))
print('CCAs con datos de salario: {}'.format(df['salario_neto_mes'].notna().sum()))
df[['ciudad', 'ccaa', 'alquiler_mes', 'salario_neto_mes']].head(10)

## 2. Asequibilidad — ¿Qué ciudades superan el umbral del 30%?

In [ ]:
# Ratio: alquiler mensual / salario neto mensual
df['ratio_alquiler_salario'] = df['alquiler_mes'] / df['salario_neto_mes']
df['pct_salario_alquiler'] = df['ratio_alquiler_salario'] * 100

# Meses de salario neto necesarios para pagar 12 meses de alquiler
df['meses_salario_para_12m_alquiler'] = (df['alquiler_mes'] * 12) / df['salario_neto_mes']

UMBRAL_30 = 30.0  # % del ingreso mensual — estándar ONU/Banco Mundial

df_sorted = df.sort_values('pct_salario_alquiler', ascending=False)

print('Umbral de asequibilidad: {}% del salario mensual neto'.format(int(UMBRAL_30)))
print('Ciudades por encima del umbral (inasequibles):')
print()

inasequibles = df_sorted[df_sorted['pct_salario_alquiler'] > UMBRAL_30]
for _, row in inasequibles.iterrows():
    ciudad = row['ciudad']
    pct = row['pct_salario_alquiler']
    meses = row['meses_salario_para_12m_alquiler']
    print('  {}: {:.1f}% del salario ({:.1f} meses de salario para cubrir el año)'.format(ciudad, pct, meses))

print()
n_inasequibles = len(inasequibles)
n_total = len(df)
print('{} de {} ciudades superan el umbral del 30% ({:.0f}%)'.format(n_inasequibles, n_total, 100*n_inasequibles/n_total))

## 3. Scatter — Salario vs Alquiler por ciudad

In [ ]:
# Colores por región
region_colors = {
    'País Vasco': '#2196F3',
    'Comunidad de Madrid': '#F44336',
    'Cataluña': '#9C27B0',
    'Islas Baleares': '#FF9800',
    'Andalucía': '#4CAF50',
    'Comunitat Valenciana': '#00BCD4',
    'Comunidad Foral de Navarra': '#607D8B',
    'Aragón': '#795548',
    'Cantabria': '#009688',
    'Castilla y León': '#8BC34A',
    'Canarias': '#FF5722',
    'Región de Murcia': '#E91E63',
    'La Rioja': '#3F51B5'
}

fig, ax = plt.subplots(figsize=(12, 7))

for _, row in df.iterrows():
    ccaa = row['ccaa']
    color = region_colors.get(ccaa, '#9E9E9E')
    ax.scatter(row['salario_neto_mes'], row['alquiler_mes'], color=color, s=90, zorder=3, edgecolors='white', linewidths=0.5)
    ax.annotate(
        row['ciudad'],
        (row['salario_neto_mes'], row['alquiler_mes']),
        textcoords='offset points',
        xytext=(5, 4),
        fontsize=7.5,
        color='#333333'
    )

# Línea del 30%: alquiler = 0.30 * salario_neto_mes
x_range = np.linspace(df['salario_neto_mes'].min() - 50, df['salario_neto_mes'].max() + 100, 100)
ax.plot(x_range, 0.30 * x_range, '--', color='red', linewidth=1.5, alpha=0.7, label='Umbral 30% (inasequible)')
ax.fill_between(x_range, 0.30 * x_range, x_range * 2, alpha=0.05, color='red')

ax.set_xlabel('Salario neto mensual CCAA (€)', fontsize=11)
ax.set_ylabel('Alquiler mensual (€)', fontsize=11)
ax.set_title('Salario vs Alquiler por ciudad — España 2024\n(zona roja = por encima del 30% del salario)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Ratio renta/salario — Barh ordenado por inasequibilidad

In [ ]:
df_plot = df.sort_values('pct_salario_alquiler', ascending=True)

colors = ['tomato' if v > UMBRAL_30 else 'steelblue' for v in df_plot['pct_salario_alquiler']]

fig, ax = plt.subplots(figsize=(11, 8))
bars = ax.barh(df_plot['ciudad'], df_plot['pct_salario_alquiler'], color=colors, edgecolor='white')

ax.axvline(x=UMBRAL_30, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label='Umbral 30%')
ax.set_xlabel('% del salario neto mensual destinado a alquiler', fontsize=11)
ax.set_title('Inasequibilidad del alquiler por ciudad — España 2024\n(rojo = supera el umbral del 30%)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, axis='x', alpha=0.3)

for bar, val in zip(bars, df_plot['pct_salario_alquiler']):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, '{:.1f}%'.format(val), va='center', fontsize=8)

blue_patch = mpatches.Patch(color='steelblue', label='Asequible (<30%)')
red_patch = mpatches.Patch(color='tomato', label='Inasequible (>30%)')
ax.legend(handles=[blue_patch, red_patch], fontsize=9, loc='lower right')

plt.tight_layout()
plt.show()

## 5. Evolución 2019 vs 2024 — 5 ciudades principales

In [ ]:
# Datos históricos conocidos (Idealista histórico + INE años anteriores)
evolucion = {
    'Madrid': {'alq_2019': 1200, 'alq_2024': 1800, 'sal_neto_mes_2019': 1650, 'sal_neto_mes_2024': 1781},
    'Barcelona': {'alq_2019': 1100, 'alq_2024': 1700, 'sal_neto_mes_2019': 1600, 'sal_neto_mes_2024': 1631},
    'Valencia': {'alq_2019': 650, 'alq_2024': 1000, 'sal_neto_mes_2019': 1250, 'sal_neto_mes_2024': 1388},
    'Sevilla': {'alq_2019': 600, 'alq_2024': 900, 'sal_neto_mes_2019': 1100, 'sal_neto_mes_2024': 1313},
    'Bilbao': {'alq_2019': 900, 'alq_2024': 1200, 'sal_neto_mes_2019': 1400, 'sal_neto_mes_2024': 1950},
}

ciudades_evol = list(evolucion.keys())
ratio_2019 = [evolucion[c]['alq_2019'] / evolucion[c]['sal_neto_mes_2019'] * 100 for c in ciudades_evol]
ratio_2024 = [evolucion[c]['alq_2024'] / evolucion[c]['sal_neto_mes_2024'] * 100 for c in ciudades_evol]

x = np.arange(len(ciudades_evol))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
bars_2019 = ax.bar(x - width/2, ratio_2019, width, label='2019', color='#90CAF9', edgecolor='white')
bars_2024 = ax.bar(x + width/2, ratio_2024, width, label='2024', color='#EF5350', edgecolor='white')

ax.axhline(y=30, color='darkred', linestyle='--', linewidth=1.5, alpha=0.7, label='Umbral 30%')
ax.set_xticks(x)
ax.set_xticklabels(ciudades_evol, fontsize=10)
ax.set_ylabel('% salario neto mensual destinado a alquiler', fontsize=10)
ax.set_title('Evolución del esfuerzo por alquiler — 2019 vs 2024\n5 ciudades principales', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, axis='y', alpha=0.3)

for bar, val in zip(bars_2019, ratio_2019):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, '{:.1f}%'.format(val), ha='center', va='bottom', fontsize=8)
for bar, val in zip(bars_2024, ratio_2024):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, '{:.1f}%'.format(val), ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

print('Incremento del esfuerzo por alquiler entre 2019 y 2024:')
for i, ciudad in enumerate(ciudades_evol):
    diff = ratio_2024[i] - ratio_2019[i]
    print('  {}: +{:.1f} puntos porcentuales'.format(ciudad, diff))

## 6. Conclusiones

### Resultados del análisis de asequibilidad

**Ciudades más inasequibles (2024):**
- **Palma de Mallorca** encabeza el ranking: combinación de salarios medios y alquileres disparados por el turismo.
- **Málaga y Barcelona** superan ampliamente el umbral del 30%, impulsadas por presión turística y golden visa.
- **Donostia-San Sebastián** es la excepción vasca: alquiler alto pero salario también alto — el ratio se modera.

**Evolución 2019–2024:**
- El esfuerzo económico por alquiler ha aumentado entre 5 y 15 puntos porcentuales en 5 años.
- Los salarios apenas han crecido un 5-8% en el período, mientras el alquiler subió entre un 40% y 60%.
- La brecha es especialmente grave en Valencia (+54% de aumento del alquiler) y Málaga (+62%).

**Umbral del 30%:**
- Según el estándar ONU/Banco Mundial, la vivienda es inasequible cuando supera el 30% del ingreso.
- En España 2024, la mayoría de grandes ciudades superan este umbral — la vivienda se ha convertido en el mayor gasto de los hogares.

**Fuentes:**
- INE Encuesta de Estructura Salarial, tabla 10882
- Idealista, Fotocasa, informes de prensa especializada 2024